# Schrödinger Equation — Physics-Informed Neural Network

The **nonlinear Schrödinger (NLS) equation** describes the evolution of a complex-valued wave function $h(t, x)$:

$$i \, h_t + \tfrac{1}{2} h_{xx} + |h|^2 h = 0, \quad x \in [-5, 5], \quad t \in [0, \pi/2]$$

Writing $h = u + iv$ (real + imaginary parts), this becomes a coupled real system:

$$f_u = -v_t + \tfrac{1}{2} u_{xx} + (u^2 + v^2)\,u = 0$$
$$f_v = \;\; u_t + \tfrac{1}{2} v_{xx} + (u^2 + v^2)\,v = 0$$

**Initial condition:** $h(0, x) = 2\,\mathrm{sech}(x)$ (i.e. $u(0,x) = 2/\cosh(x)$, $v(0,x) = 0$)

**Boundary conditions:** Periodic — $h(-5, t) = h(5, t)$ and $h_x(-5, t) = h_x(5, t)$

The PINN outputs both $u$ and $v$ simultaneously, with a four-component loss enforcing the initial condition, the periodic BCs (values and first derivatives), and both PDE residuals.

> Reference: *Raissi, Perdikaris & Karniadakis — Physics-informed neural networks, J. Comput. Phys. 378 (2019)*

In [ ]:
import torch
import torch.nn as nn
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Network Architecture

Unlike the Burgers PINN (scalar output), the Schrödinger PINN produces **two outputs** — the real part $u$ and imaginary part $v$ of the wave function — since the equation is complex-valued.

**Architecture:** Linear(2→40) + 7×[Linear(40→40) + Tanh] + Linear(40→**2**)

- Input: $(t, x)$
- Output: $(\hat{u}(t, x),\; \hat{v}(t, x))$
- Tanh activations: required for smooth second-order derivatives via autograd

In [ ]:
class PINNs(nn.Module):
    """Fully-connected neural network that approximates the complex wave function h = u + iv.

    Outputs two values (real part u, imaginary part v) simultaneously.
    Architecture: Linear(2→40) + 7×[Linear(40→40) + Tanh] + Linear(40→2).
    Tanh is used throughout to guarantee C∞ smoothness, required for second-order autograd.
    """

    NUM_NEURONS = 40  # neurons per hidden layer
    NUM_LAYERS  = 9   # 1 input projection + 7 hidden + 1 output

    def __init__(self):
        super().__init__()
        layers = []

        # Input projection: (t, x) → hidden dimension
        layers.append(nn.Linear(2, self.NUM_NEURONS))
        layers.append(nn.Tanh())

        # Hidden layers
        for _ in range(self.NUM_LAYERS - 1):
            layers.append(nn.Linear(self.NUM_NEURONS, self.NUM_NEURONS))
            layers.append(nn.Tanh())

        # Output head: hidden → (u, v) — two real outputs representing the complex solution
        layers.append(nn.Linear(self.NUM_NEURONS, 2))

        self.network = nn.Sequential(*layers)

    def forward(self, t, x):
        """Forward pass.

        Args:
            t: Time coordinates of shape (N, 1).
            x: Spatial coordinates of shape (N, 1).

        Returns:
            u: Real part of the wave function, shape (N, 1).
            v: Imaginary part of the wave function, shape (N, 1).
        """
        inputs  = torch.cat([t, x], dim=1)  # (N, 2)
        outputs = self.network(inputs)       # (N, 2)
        u = outputs[:, 0:1]                 # real part
        v = outputs[:, 1:2]                 # imaginary part
        return u, v

## 2. PDE Residual

The coupled residuals enforce both equations of the NLS system at interior collocation points:

$$f_u = -v_t + \tfrac{1}{2} u_{xx} + (u^2 + v^2)\,u$$
$$f_v = \;\; u_t + \tfrac{1}{2} v_{xx} + (u^2 + v^2)\,v$$

Six partial derivatives are required: $u_t, u_x, u_{xx}, v_t, v_x, v_{xx}$ — all computed via autograd.

In [ ]:
def physics_residual(model, t, x):
    """Evaluate the NLS PDE residuals at collocation points.

    Computes the two coupled residuals:
        f_u = -v_t + 0.5·u_xx + (u² + v²)·u   (should be 0)
        f_v =  u_t + 0.5·v_xx + (u² + v²)·v   (should be 0)

    All six required partial derivatives are computed via autograd with
    create_graph=True so that the residual loss is differentiable w.r.t. weights.

    Args:
        model: The PINN model returning (u, v).
        t: Time coordinates of shape (N, 1). Detached and re-enabled for grad.
        x: Spatial coordinates of shape (N, 1). Detached and re-enabled for grad.

    Returns:
        f_u: Real-part residual, shape (N, 1).
        f_v: Imaginary-part residual, shape (N, 1).
    """
    t = t.clone().detach().requires_grad_(True)
    x = x.clone().detach().requires_grad_(True)

    u, v = model(t, x)

    # ── Derivatives of the real part u ────────────────────────────────────────
    u_t = torch.autograd.grad(
        u, t, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True
    )[0]
    u_x = torch.autograd.grad(
        u, x, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True
    )[0]
    u_xx = torch.autograd.grad(
        u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True
    )[0]

    # ── Derivatives of the imaginary part v ───────────────────────────────────
    v_t = torch.autograd.grad(
        v, t, grad_outputs=torch.ones_like(v), create_graph=True, retain_graph=True
    )[0]
    v_x = torch.autograd.grad(
        v, x, grad_outputs=torch.ones_like(v), create_graph=True, retain_graph=True
    )[0]
    v_xx = torch.autograd.grad(
        v_x, x, grad_outputs=torch.ones_like(v_x), create_graph=True, retain_graph=True
    )[0]

    # ── NLS coupling term: |h|² = u² + v² ────────────────────────────────────
    h_sq = u ** 2 + v ** 2

    # ── Coupled residuals ─────────────────────────────────────────────────────
    f_u = -v_t + 0.5 * u_xx + h_sq * u   # from Re(NLS) = 0
    f_v =  u_t + 0.5 * v_xx + h_sq * v   # from Im(NLS) = 0

    return f_u, f_v

## 3. Periodic Boundary Conditions

The NLS equation is solved on a periodic domain $x \in [-5, 5]$. Periodicity requires:

$$u(-5, t) = u(5, t), \quad v(-5, t) = v(5, t)$$
$$u_x(-5, t) = u_x(5, t), \quad v_x(-5, t) = v_x(5, t)$$

All four constraints are enforced via a dedicated boundary loss term. Note that matching derivatives (not just values) is necessary to prevent the network from learning a discontinuous periodic extension.

In [ ]:
def boundary_residual(model, t_b):
    """Compute the periodic boundary mismatch at x = −5 and x = +5.

    Enforces four periodicity constraints:
        u(−5, t) = u(5, t)        — real part values match
        v(−5, t) = v(5, t)        — imaginary part values match
        u_x(−5, t) = u_x(5, t)   — real part derivatives match
        v_x(−5, t) = v_x(5, t)   — imaginary part derivatives match

    Args:
        model: The PINN model returning (u, v).
        t_b: Boundary time coordinates of shape (N_bc, 1).

    Returns:
        u_L, v_L: Solution values at x = −5, each shape (N_bc, 1).
        u_R, v_R: Solution values at x = +5, each shape (N_bc, 1).
        u_x_L, v_x_L: Spatial derivatives at x = −5, each shape (N_bc, 1).
        u_x_R, v_x_R: Spatial derivatives at x = +5, each shape (N_bc, 1).
    """
    t_b    = t_b.clone().detach().requires_grad_(True)
    x_left  = (-5.0 * torch.ones_like(t_b)).requires_grad_(True)
    x_right = ( 5.0 * torch.ones_like(t_b)).requires_grad_(True)

    # ── Left boundary (x = −5) ────────────────────────────────────────────────
    u_L, v_L = model(t_b, x_left)
    u_x_L = torch.autograd.grad(
        u_L, x_left, grad_outputs=torch.ones_like(u_L),
        create_graph=True, retain_graph=True
    )[0]
    v_x_L = torch.autograd.grad(
        v_L, x_left, grad_outputs=torch.ones_like(v_L),
        create_graph=True, retain_graph=True
    )[0]

    # ── Right boundary (x = +5) ───────────────────────────────────────────────
    u_R, v_R = model(t_b, x_right)
    u_x_R = torch.autograd.grad(
        u_R, x_right, grad_outputs=torch.ones_like(u_R),
        create_graph=True, retain_graph=True
    )[0]
    v_x_R = torch.autograd.grad(
        v_R, x_right, grad_outputs=torch.ones_like(v_R),
        create_graph=True, retain_graph=True
    )[0]

    return u_L, v_L, u_R, v_R, u_x_L, v_x_L, u_x_R, v_x_R

## 4. Loss Function

The total loss has three components:

$$\mathcal{L} = \underbrace{\frac{1}{N_{IC}}\sum \left[(u - u_{IC})^2 + (v - v_{IC})^2\right]}_{\text{MSE}_a \;:\; \text{initial condition}} + \underbrace{\frac{1}{N_{BC}}\sum \left[(u_L - u_R)^2 + (v_L - v_R)^2 + (u_{x,L} - u_{x,R})^2 + (v_{x,L} - v_{x,R})^2\right]}_{\text{MSE}_b \;:\; \text{periodic BC}} + \underbrace{\frac{1}{N_f}\sum (f_u^2 + f_v^2)}_{\text{MSE}_f \;:\; \text{PDE residual}}$$

In [ ]:
def compute_loss(model, t_ic, x_ic, u_ic, v_ic, t_b, t_f, x_f):
    """Compute the composite PINN loss for the NLS equation.

    Args:
        model: The PINN model returning (u, v).
        t_ic, x_ic: Initial condition coordinates at t = 0, each (N_ic, 1).
        u_ic, v_ic: Known initial values (real and imaginary parts), each (N_ic, 1).
        t_b: Time coordinates for boundary condition enforcement, shape (N_bc, 1).
        t_f, x_f: Collocation point coordinates, each (N_f, 1).

    Returns:
        loss: Total loss (scalar).
        mse_a: Initial condition loss.
        mse_b: Periodic boundary condition loss.
        mse_f: PDE residual loss.
    """
    # ── Initial condition loss ─────────────────────────────────────────────────
    u_pred, v_pred = model(t_ic, x_ic)
    mse_a = torch.mean((u_pred - u_ic) ** 2 + (v_pred - v_ic) ** 2)

    # ── Periodic boundary condition loss ──────────────────────────────────────
    u_L, v_L, u_R, v_R, u_x_L, v_x_L, u_x_R, v_x_R = boundary_residual(model, t_b)
    mse_b = torch.mean(
        (u_L - u_R) ** 2 + (v_L - v_R) ** 2 +       # value periodicity
        (u_x_L - u_x_R) ** 2 + (v_x_L - v_x_R) ** 2 # derivative periodicity
    )

    # ── PDE residual loss ─────────────────────────────────────────────────────
    f_u, f_v = physics_residual(model, t_f, x_f)
    mse_f = torch.mean(f_u ** 2 + f_v ** 2)

    loss = mse_a + mse_b + mse_f
    return loss, mse_a, mse_b, mse_f

## 5. Training Data

| Set | Count | Description |
|---|---|---|
| Initial condition | 50 | $u(0,x) = 2\,\mathrm{sech}(x)$, $v(0,x) = 0$ on a uniform grid |
| Boundary (time) | 50 | Random $t \sim \text{Uniform}[0, \pi/2]$ at $x = \pm 5$ |
| Collocation | 20 000 | Latin Hypercube Sampling over $(t, x) \in [0, \pi/2] \times [-5, 5]$ |

**Why Latin Hypercube Sampling (LHS)?**  
LHS partitions each dimension into equal intervals and places exactly one sample per interval, guaranteeing more uniform coverage than pure random sampling. This is especially important for a 20 000-point collocation set where clustering would leave large regions poorly supervised.

In [ ]:
from scipy.stats import qmc

# ── Point counts ──────────────────────────────────────────────────────────────
N_ic = 50     # initial condition points (uniform grid in x)
N_bc = 50     # boundary time points (used for both x = −5 and x = +5)
N_f  = 20000  # collocation points (LHS for uniform domain coverage)

# ── Initial condition: t = 0, h(0, x) = 2·sech(x) ────────────────────────────
x_ic = torch.linspace(-5, 5, N_ic).unsqueeze(1)   # uniform grid in [-5, 5]
t_ic = torch.zeros(N_ic, 1)
u_ic = 2.0 / torch.cosh(x_ic)   # real part: 2·sech(x)
v_ic = torch.zeros(N_ic, 1)     # imaginary part: 0

# ── Boundary time points: t ~ Uniform[0, π/2] ─────────────────────────────────
# These are used to evaluate the model at both x = −5 and x = +5
t_bc = torch.rand(N_bc, 1) * (torch.pi / 2)

# ── Collocation points: Latin Hypercube Sampling ──────────────────────────────
# LHS ensures uniform coverage — each marginal dimension is stratified
sampler = qmc.LatinHypercube(d=2)
sample  = sampler.random(N_f)

t_f = torch.tensor(sample[:, 0:1] * np.pi / 2, dtype=torch.float32)  # t ∈ [0, π/2]
x_f = torch.tensor(sample[:, 1:2] * 10 - 5,    dtype=torch.float32)  # x ∈ [−5, 5]

# Move all tensors to the target device
x_ic = x_ic.to(device)
t_ic = t_ic.to(device)
u_ic = u_ic.to(device)
v_ic = v_ic.to(device)
t_bc = t_bc.to(device)
t_f  = t_f.to(device)
x_f  = x_f.to(device)

print(f"Initial condition points : {N_ic}")
print(f"Boundary time points     : {N_bc}")
print(f"Collocation points       : {N_f}  (Latin Hypercube)")

## 6. Two-Phase Training

Same strategy as the Burgers PINN:

| Phase | Optimizer | Iterations | Purpose |
|---|---|---|---|
| 1 | Adam, lr = 1e-3 | 5 000 epochs | Fast convergence from random init |
| 2 | L-BFGS, Strong Wolfe | up to 50 000 | High-precision refinement |

The Schrödinger case is harder than Burgers — the loss has more components (IC + periodic BC + two coupled PDE residuals) and the solution is complex-valued — so it takes more L-BFGS iterations to converge.

In [ ]:
# ── Phase 1: Adam ─────────────────────────────────────────────────────────────
model          = PINNs().to(device)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Phase 1: Adam")
for epoch in range(5000):
    optimizer_adam.zero_grad()
    loss, mse_a, mse_b, mse_f = compute_loss(
        model, t_ic, x_ic, u_ic, v_ic, t_bc, t_f, x_f
    )
    loss.backward()
    optimizer_adam.step()

    if epoch % 500 == 0:
        print(f"  Epoch {epoch:5d} | Loss: {loss.item():.6f} | "
              f"MSE_a: {mse_a.item():.6f} | "
              f"MSE_b: {mse_b.item():.6f} | "
              f"MSE_f: {mse_f.item():.6f}")

# ── Phase 2: L-BFGS ───────────────────────────────────────────────────────────
optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(),
    lr=1.0,
    max_iter=50000,
    max_eval=50000,
    tolerance_grad=1e-7,
    tolerance_change=1e-9,
    history_size=50,
    line_search_fn="strong_wolfe"
)

iteration = [0]  # mutable counter for the closure

def closure():
    optimizer_lbfgs.zero_grad()
    loss, mse_a, mse_b, mse_f = compute_loss(
        model, t_ic, x_ic, u_ic, v_ic, t_bc, t_f, x_f
    )
    loss.backward()

    iteration[0] += 1
    if iteration[0] % 500 == 0:
        print(f"  L-BFGS iter {iteration[0]:5d} | Loss: {loss.item():.6f} | "
              f"MSE_a: {mse_a.item():.6f} | "
              f"MSE_b: {mse_b.item():.6f} | "
              f"MSE_f: {mse_f.item():.6f}")
    return loss

print("\nPhase 2: L-BFGS")
optimizer_lbfgs.step(closure)
print(f"\nFinal loss: {closure().item():.6f}")